# Lab 03: CSI 300 Linear Regression

This notebook uses the CSI 300 daily dataset exported by `get_data.py` and completes:

1. Data loading and preprocessing
2. Feature selection
3. Train/test split
4. Linear regression training
5. Prediction and evaluation

Target variable: closing price.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False
sns.set_theme(style="whitegrid")


## 1. Data Loading And Preprocessing


In [ ]:
data_dir = Path("code")
csv_candidates = sorted(data_dir.glob("*.csv"))

if csv_candidates:
    data_path = csv_candidates[0]
    raw_df = pd.read_csv(data_path, encoding="utf-8-sig")
else:
    raise FileNotFoundError("No CSV file was found under the code directory.")

date_col, target_col, open_col, high_col, low_col, volume_col, pct_col = raw_df.columns.tolist()

print(f"Data file: {data_path}")
print(f"Raw shape: {raw_df.shape}")
raw_df.head()


In [ ]:
df = raw_df.copy()

df[date_col] = pd.to_datetime(df[date_col])
df[volume_col] = df[volume_col].astype(str).str.replace("K", "", regex=False).astype(float) * 1000
df[pct_col] = pd.to_numeric(df[pct_col].astype(str).str.replace("%", "", regex=False), errors="coerce")
df = df.sort_values(date_col).reset_index(drop=True)

missing_before = df.isna().sum()
df[pct_col] = df[pct_col].fillna(df[pct_col].median())
missing_after = df.isna().sum()

rows_before = len(df)
duplicate_count = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
rows_after = len(df)

df["year"] = df[date_col].dt.year
df["month"] = df[date_col].dt.month
df["day"] = df[date_col].dt.day
df["weekday"] = df[date_col].dt.weekday + 1

numeric_cols = [target_col, open_col, high_col, low_col, volume_col, pct_col]
q1 = df[numeric_cols].quantile(0.25)
q3 = df[numeric_cols].quantile(0.75)
iqr = q3 - q1
lower_bounds = q1 - 1.5 * iqr
upper_bounds = q3 + 1.5 * iqr
outlier_counts = ((df[numeric_cols] < lower_bounds) | (df[numeric_cols] > upper_bounds)).sum()
df[numeric_cols] = df[numeric_cols].clip(lower=lower_bounds, upper=upper_bounds, axis=1)

summary_df = pd.DataFrame({
    "item": ["rows_before", "rows_after_drop_duplicates", "duplicate_rows_removed", "missing_values_before", "missing_values_after"],
    "value": [rows_before, rows_after, duplicate_count, int(missing_before.sum()), int(missing_after.sum())],
})

print("Preprocessing summary:")
display(summary_df)
print("Outlier counts detected by IQR:")
display(outlier_counts.rename("outlier_count").to_frame())
print("Cleaned data preview:")
df.head()


## 2. Feature Selection


In [ ]:
candidate_features = [open_col, high_col, low_col, volume_col, pct_col, "year", "month", "day", "weekday"]

corr_df = df[candidate_features + [target_col]].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_df, annot=True, cmap="RdBu_r", center=0, fmt=".2f")
plt.title("Correlation Matrix")
plt.show()

corr_with_target = corr_df[target_col].drop(target_col).abs().sort_values(ascending=False)
corr_with_target.to_frame(name="abs_corr_with_target")


In [ ]:
high_corr_threshold = 0.95
dominant_ratio_threshold = 0.95
target_corr_threshold = 0.10

feature_corr = df[candidate_features].corr().abs()
upper = feature_corr.where(np.triu(np.ones(feature_corr.shape), k=1).astype(bool))

high_corr_records = []
drop_high_corr = set()
for col in upper.columns:
    related_rows = upper.index[upper[col] > high_corr_threshold].tolist()
    for row in related_rows:
        keep_feature = row if corr_with_target[row] >= corr_with_target[col] else col
        drop_feature = col if keep_feature == row else row
        high_corr_records.append({
            "feature_1": row,
            "feature_2": col,
            "correlation": upper.loc[row, col],
            "keep": keep_feature,
            "drop": drop_feature,
        })
        drop_high_corr.add(drop_feature)

high_corr_result = pd.DataFrame(high_corr_records)
if high_corr_result.empty:
    high_corr_result = pd.DataFrame(columns=["feature_1", "feature_2", "correlation", "keep", "drop"])

dominant_ratio = df[candidate_features].apply(lambda s: s.value_counts(normalize=True, dropna=False).iloc[0])
drop_dominant = dominant_ratio[dominant_ratio > dominant_ratio_threshold].index.tolist()

# pct_col is derived from the close price and is excluded to avoid target leakage.
leakage_features = [pct_col]

selected_features = []
feature_decisions = []
for feature in candidate_features:
    if feature in drop_high_corr:
        decision = "drop_high_correlation"
    elif feature in drop_dominant:
        decision = "drop_dominant_single_value"
    elif feature in leakage_features:
        decision = "drop_target_leakage"
    elif corr_with_target[feature] < target_corr_threshold:
        decision = "drop_low_target_correlation"
    else:
        decision = "keep"
        selected_features.append(feature)

    feature_decisions.append({
        "feature": feature,
        "abs_corr_with_target": corr_with_target[feature],
        "dominant_ratio": dominant_ratio[feature],
        "decision": decision,
    })

feature_selection_df = pd.DataFrame(feature_decisions).sort_values("abs_corr_with_target", ascending=False)

print("Highly correlated feature pairs:")
display(high_corr_result)
print("Feature selection decisions:")
display(feature_selection_df)
print(f"Selected features: {selected_features}")


## 3. Train/Test Split


In [ ]:
X = df[selected_features].copy()
y = df[target_col].copy()

split_ratio = 0.8
split_index = int(len(df) * split_ratio)

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()
y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

split_summary = pd.DataFrame({
    "dataset": ["train", "test"],
    "samples": [len(X_train), len(X_test)],
    "ratio": [f"{len(X_train) / len(df):.1%}", f"{len(X_test) / len(df):.1%}"],
    "date_range": [
        f"{df.loc[0, date_col].date()} to {df.loc[split_index - 1, date_col].date()}",
        f"{df.loc[split_index, date_col].date()} to {df.loc[len(df) - 1, date_col].date()}",
    ],
})

print("Chronological split (80% train / 20% test):")
split_summary


## 4. Linear Regression Training


In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

coef_df = pd.DataFrame({
    "feature": selected_features,
    "coefficient": linear_model.coef_,
}).sort_values("coefficient", key=np.abs, ascending=False)

print(f"Intercept: {linear_model.intercept_:.4f}")
coef_df


## 5. Prediction And Evaluation


In [ ]:
y_pred = linear_model.predict(X_test)

mae_value = mean_absolute_error(y_test, y_pred)
rmse_value = np.sqrt(mean_squared_error(y_test, y_pred))
r2_value = r2_score(y_test, y_pred)

evaluation_df = pd.DataFrame({
    "metric": ["MAE", "RMSE", "R2"],
    "value": [mae_value, rmse_value, r2_value],
})

result_df = pd.DataFrame({
    date_col: df.loc[y_test.index, date_col].values,
    "actual_close": y_test.values,
    "predicted_close": y_pred,
})
result_df["absolute_error"] = (result_df["actual_close"] - result_df["predicted_close"]).abs()

print("Evaluation metrics:")
display(evaluation_df)
print("Prediction preview:")
display(result_df.head(10))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(result_df[date_col], result_df["actual_close"], label="actual", linewidth=2)
axes[0].plot(result_df[date_col], result_df["predicted_close"], label="predicted", linewidth=2, alpha=0.85)
axes[0].set_title("Actual vs Predicted Close Price")
axes[0].set_xlabel("date")
axes[0].set_ylabel("close price")
axes[0].tick_params(axis="x", rotation=45)
axes[0].legend()

axes[1].scatter(y_test, y_pred, alpha=0.75)
line_min = min(y_test.min(), y_pred.min())
line_max = max(y_test.max(), y_pred.max())
axes[1].plot([line_min, line_max], [line_min, line_max], color="red", linestyle="--")
axes[1].set_title("Actual vs Predicted Scatter")
axes[1].set_xlabel("actual close")
axes[1].set_ylabel("predicted close")

plt.tight_layout()
plt.show()


## Conclusion

- The dataset was cleaned through missing-value handling, duplicate-row removal, and IQR-based outlier clipping.
- Feature selection removed redundant high-correlation features, low-information features, and the leakage-prone percentage-change feature.
- The final selected features are expected to be the daily high price, trading volume, and year.
- The fitted linear regression model is evaluated by R2. A value closer to 1 indicates a better fit.
